# Randomized benchmarking

This notebook shows a compact randomized benchmarking workflow: prepare a
good single-qubit gate set, run standard RB, and then estimate specific
gate errors with interleaved RB.

## 1. Create an `Experiment`

In [ ]:
import numpy as np

import qubex as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    qubits=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

In [ ]:
Q0 = exp.qubit_labels[0]

print("target qubit:", Q0)

## 2. Connect and prepare DRAG-calibrated pulses

RB is most useful after the drive pulses are already in good shape.

In [ ]:
exp.connect()

rabi_result = exp.obtain_rabi_params(targets=Q0, n_shots=512, plot=True)
drag_hpi_result = exp.calibrate_drag_hpi_pulse(
    targets=Q0,
    n_iterations=2,
    n_shots=512,
    plot=True,
)
drag_pi_result = exp.calibrate_drag_pi_pulse(
    targets=Q0,
    n_iterations=2,
    n_shots=512,
    plot=True,
)

## 3. Inspect the calibrated gate primitives

In [ ]:
exp.drag_hpi_pulse[Q0].plot()
exp.drag_pi_pulse[Q0].plot()

repeat_drag_hpi = exp.repeat_sequence({Q0: exp.drag_hpi_pulse[Q0]})
repeat_drag_pi = exp.repeat_sequence({Q0: exp.drag_pi_pulse[Q0]})

## 4. Run standard randomized benchmarking

In [ ]:
rb_result = exp.randomized_benchmarking(
    Q0,
    n_cliffords_range=np.arange(0, 401, 40),
    n_trials=30,
    n_shots=512,
    plot=True,
    save_image=False,
)

## 5. Run interleaved RB for `X90` and `X180`

Interleaved RB isolates the error contribution of one gate inside the same
randomized-sequence framework.

In [ ]:
irb_x90 = exp.interleaved_randomized_benchmarking(
    Q0,
    interleaved_clifford="X90",
    n_cliffords_range=np.arange(0, 401, 40),
    n_trials=30,
    n_shots=512,
    plot=True,
    save_image=False,
)

irb_x180 = exp.interleaved_randomized_benchmarking(
    Q0,
    interleaved_clifford="X180",
    n_cliffords_range=np.arange(0, 401, 40),
    n_trials=30,
    n_shots=512,
    plot=True,
    save_image=False,
)

## 6. Optionally save the updated calibration note

In [ ]:
# exp.calib_note.save()